In [1]:
import os
import sys

# Bind rpy2 to the R installation paired with the active Python environment.
_active_env_bin = os.path.dirname(sys.executable)
os.environ['PATH'] = _active_env_bin + os.pathsep + os.environ.get('PATH', '')
_active_r_home = os.path.join(sys.prefix, 'lib', 'R')
if os.path.isdir(_active_r_home):
    os.environ['R_HOME'] = _active_r_home

from WERM.signal_priors import SparseGaussianSignal, SparseDiffSignal
import WERM.model
import WERM.performance_measures
import importlib
import WERM.comparison
importlib.reload(WERM.comparison)
import WERM.estimators
from WERM.config import η_to_ψ
import numpy as np
import jax.numpy as jnp
import jax
jax.config.update('jax_enable_x64', True)
import numpy.random as nprandom
import matplotlib.pyplot as plt
# Embed figures in executed notebook output even on a headless compute node.
plt.switch_backend("module://matplotlib_inline.backend_inline")
from functools import partial
from tqdm.auto import tqdm
from WERM.experiment_utils import data_hash as _data_hash, evaluate_changepoints as _evaluate_changepoints, summarize_setting as _summarize_setting, trial_rng as _trial_rng, weighted_erm_joint_cv as _weighted_erm_joint_cv
import time
from joblib import Parallel, delayed
import sys
import pickle
import os
os.environ['JAX_ENABLE_X64']='True'

# Set a random seed for reproducibility
random_seed = 46
SMOKE_MODE = os.environ.get('WERM_SMOKE') == '1'

# --- Helper Function for a Single Parallel Job ---
def run_single_trial(params, p, L, fixed_noise_scale, fractional_chgpt_locations, M, gamma_candidates, method_configs):
    """
    Executes a single simulation trial for a given delta, running multiple methods.
    This function is designed to be called in parallel by joblib.
    """
    i, delta, j = params # delta is now the iterating parameter

    rng = _trial_rng("M1b-alpha-1.0-gaussian", i, j)

    # --- Setup based on delta and fixed noise_scale ---
    n = int(delta * p)
    true_chgpt_locations_fn = lambda n_len: (n_len * fractional_chgpt_locations).astype(int)

    # Sparse Difference Signal Prior
    # α = 0.5 # fraction of nonzeros in change vector
    # var_β_1 = 8
    # σ_w = np.sqrt(400*delta) # standard deviation of the signal perturbation
    # signal_prior = SparseDiffSignal(var_β_1, σ_w, α, L)
    
    κ = 1.0
    α = 1.0 # Fully dense signal
    σ_l_arr = κ * np.sqrt(delta) * np.ones(L)  # Signal standard deviations
    signal_prior = SparseGaussianSignal(α, delta, σ_l_arr)

    # --- Data Generation for this trial ---
    B̃ = signal_prior.sample(p, rng=rng)
    # print("B̃ shape:", B̃.shape) # Keep for debugging if needed
    X = rng.normal(0, np.sqrt(1/n), (n, p))
    # ε = nprandom.normal(0, 1, size=(n,)) * fixed_noise_scale # Using fixed_noise_scale here
    epsilon = rng.normal(0.0, fixed_noise_scale, size=n)
    ε = epsilon  # Preserve the response-generation notation used below.


    X = X @ np.linalg.cholesky(Sigma).T
    true_chgpts = true_chgpt_locations_fn(n)
    C_true = η_to_ψ(true_chgpts, n) if true_chgpts.size > 0 else np.zeros(n).astype(int)

    X_scaled = X * np.sqrt(n)
    B̃_scaled = B̃ / np.sqrt(n)
    Θ = X_scaled @ B̃_scaled
    y = jax.vmap(WERM.model.q_linear_model, in_axes=(0, 0, 0))(Θ, C_true, ε)
    y_standardized = np.array(y / np.std(y))  # Standardize y

    # --- Run Estimators for all configured methods ---
    trial_results = {'data_hash': _data_hash(X, y)}
    methods_status = {method_name: False for method_name in method_configs.keys()}

    for method_name, method_func_info in method_configs.items():
        try:
            # print(f"Running {method_name} on params: ", params) # Keep for debugging if needed
            method_start = time.time()
            cp = np.array([]) # Initialize changepoints

            if method_name == 'McScan':
                # print(f"Input shapes: X_scaled: {np.array(X_scaled).shape}, y: {np.array(y).shape}") # Keep for debugging if needed
                # print(f"New input shapes: X_scaled: {np.array(X_scaled).shape}, y: {np.array(y).reshape((n, 1)).shape}") # Keep for debugging if needed
                cp = np.asarray(WERM.comparison.run_mcscan(np.array(X_scaled), y_standardized.flatten(), num_chgpts = None, method='auto'))
                # print(f"McScan res: {cp}") # Keep for debugging if needed
                cp = np.array([c for c in cp if c != -1], dtype=int)  # Remove -1s and convert to 0-indexed
            elif method_name == 'DCDP':
                cp = np.asarray(WERM.comparison.DCDP(np.array(X_scaled), y_standardized))
                # print(f"True changepoints: {true_chgpts}, DCDP changepoints: {cp}") # Keep for debugging if needed
            elif method_name == 'DPDU':
                if delta <= 7.0: 
                    cp = np.asarray(WERM.comparison.DPDU(np.array(X_scaled), y_standardized, λs = [0.01, 0.1, 1, 2], ζs = [10.0, 15.0, 20.0]))
                    cp = np.array([c for c in cp if c != -1], dtype=int)  # Remove -1s and convert to 0-indexed
                else: 
                    cp = np.array([])
            elif method_name == 'MOSEG':
                res = WERM.comparison.MOSEG(np.array(X_scaled), y_standardized, ms=True, num_chgpts = None)
                cp = res[1]  # Extract the refined estimate; R NULL means no change found.
                if cp is WERM.comparison.rpy2.rinterface.NULL:
                    cp = np.array([], dtype=int)
                else:
                    cp = np.array([c for c in cp if c != -1], dtype=int)
            elif method_name == 'charcoal':
                cp = WERM.comparison.run_charcoal(np.array(X_scaled), np.array(y))
            elif method_name == 'WERM-CV':
                Delta = lambda n_len: 1
                # Run cross-validation to find the best regularisation penalty parameter
                def WERM_l1_reg_paramCV(X_train, y_train, l1_reg_param):
                    """Cross-validation function for WERM with L1 regularization, tested for fixed num chgpts."""

                    def P_func_(beta):
                        return l1_reg_param * jnp.sum(jnp.abs(beta))

                    return WERM.estimators.weighted_erm(
                        X=X_train, y=y_train, M=M, L=L, Delta_n=Delta(n), gamma=l1_reg_param,
                        L_is_upper_bound=False, return_B_hat=False,
                        brute_force=False, P_func=P_func_,
                        coefficient_loss="least_squares", coefficient_lambda=l1_reg_param,
                        
                    )

                # Perform cross-validation to find the best gamma
                best_l1_reg, best_model_output = WERM.estimators.chgpt_k_fold_cv(
                    f=WERM_l1_reg_paramCV,
                    X=X,
                    y=y,
                    gamma_candidates=[1e-4, 1e-3, 1e-2],
                    M_scorer=M_scorer,
                    P_scorer=None,
                    k=5,
                    n_jobs=1
                )

                P_func = lambda beta: best_l1_reg * jnp.sum(jnp.abs(beta))
                cp = WERM.estimators.weighted_erm(
                    X=X, y=y, M=M, L=L, Delta_n=Delta(n), gamma=0.0,
                    L_is_upper_bound=False, return_B_hat=False,
                    brute_force=False, P_func=P_func,
                    coefficient_loss="least_squares", coefficient_lambda=best_l1_reg,
                    
                )
            elif method_name == 'WERM-L1-jointCV':
                delta_fraction = 1/20
                (best_segments, best_C), cp = _weighted_erm_joint_cv(
                    X=X_scaled, y=np.asarray(y), M=M, M_scorer=M_scorer,
                    Lmax=L, delta_fraction=delta_fraction, standardize_y=True,
                    l1_constants=(0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0),
                    num_folds=5, n_jobs=1,
                    coefficient_loss="least_squares", 
                )
                trial_results['selected_segments_WERM-L1-jointCV'] = int(best_segments)
                trial_results['selected_num_chgpts_WERM-L1-jointCV'] = int(best_segments - 1)
                trial_results['selected_l1_constant_WERM-L1-jointCV'] = float(best_C)
                trial_results['selected_l1_lambda_WERM-L1-jointCV'] = float(
                    best_C * np.sqrt(2 * np.log(p) / n)
                )

            elif method_name.startswith('WERM-unp-'):
                # Identical data and absolute held-out score; only k changes.
                num_cv_folds = int(method_name.rsplit('-', 1)[1].replace('fold', ''))
                (best_segments, _), cp = _weighted_erm_joint_cv(
                    X=X_scaled, y=y_standardized, M=M, M_scorer=M_scorer,
                    Lmax=L, delta_fraction=1/20, l1_constants=(0.0,),
                    num_folds=num_cv_folds, n_jobs=1, standardize_y=False,
                    selection_strategy='exhaustive', penalty_norm='none',
                    coefficient_loss='least_squares', 
                    coefficient_solver_options={'tol': 1e-5, 'max_iter': 50000},
                )
            elif method_name == 'WERM-fixed':
                fixed_gamma_value = 1e-3 # Example fixed gamma value
                P_func_fixed = lambda beta: fixed_gamma_value * jnp.sum(jnp.abs(beta))
                cp = WERM.estimators.weighted_erm(
                    X=X, y=y, M=M, L=L, Delta_n=1, gamma=0.0, # Delta_n=1 is a placeholder, adjust if needed
                    L_is_upper_bound=False, return_B_hat=False,
                    brute_force=False, P_func=P_func_fixed
                )
            else:
                print(f"Method {method_name} not recognized. Skipping.")
                continue

            method_time = time.time() - method_start
            evaluation = _evaluate_changepoints(cp, true_chgpts, n)
            raw_cp = evaluation['raw_cp']
            trial_results[f'runtime_{method_name}'] = method_time
            trial_results[f'raw_cp_{method_name}'] = None if raw_cp is None else raw_cp.copy()
            trial_results[f'status_{method_name}'] = evaluation['status']
            haus_value = evaluation['haus']
            if evaluation['status'] == 'empty_estimate':
                haus_value = np.nan
            trial_results[f'haus_{method_name}'] = haus_value
            trial_results[f'size_{method_name}'] = evaluation['size']
            trial_results[f'chgpts_{method_name}'] = np.array([]) if raw_cp is None else raw_cp.copy()
            methods_status[method_name] = evaluation['status'] != 'not_applicable'

        except Exception as e:
            print(f"Trial {j+1} for delta={delta:.1f}: {method_name} failed with error: {e}")
            raise

    # Store additional info for histogram plotting
    trial_results['n'] = n
    trial_results['true_chgpts'] = true_chgpts.copy()
    trial_results['fractional_true_chgpts'] = fractional_chgpt_locations.copy()
    trial_results['methods_status'] = methods_status


    return i, j, trial_results

# --- Main Script Setup ---

# Simulation Parameters
p = 200
L = 4
fixed_noise_scale = np.sqrt(0.1) # Noise scale is now fixed
delta_list = np.arange(2.0, 10.0, 1.0) # Iterate over delta
num_trials = 100
fractional_chgpt_locations = np.array([0.35, 0.65])
n_jobs = max(1, min(int(os.environ.get('WERM_N_JOBS', '36')), 36)) # Always try to run 36 jobs
if SMOKE_MODE:
    delta_list = np.array([3.0])
    num_trials = 1
    n_jobs = max(1, min(int(os.environ.get('WERM_N_JOBS', '2')), 36))

r = 0.4 # Charcoal requires i.i.d covariates
Sigma = np.eye(p)
for i_cov in range(p):
    for j_cov in range(p):
        if i_cov != j_cov:
            Sigma[i_cov, j_cov] = np.power(r, abs(i_cov - j_cov))

# def sample_corr(p, df=10, seed=None):
#     """Sample a random correlation matrix using the Haar measure."""
#     rng = np.random.default_rng(seed)
#     Q, _ = np.linalg.qr(rng.standard_normal((p, p)))           # Haar‐orthogonal
#     λ = rng.chisquare(df, size=p)                              # χ² eigenvalues
#     B = Q.T @ np.diag(λ) @ Q                                   # SPD matrix
#     D = np.sqrt(np.diag(B))
#     return B / np.outer(D, D)

# Sigma = sample_corr(p, df=10, seed=random_seed) 

# --- Define Methods to be Evaluated ---
FINAL_METHOD_NAMES = ['McScan', 'WERM-unp-10fold']
method_catalog = {
    "McScan": {
        "name": "McScan",
        "label": "McScan",
        "color": "#AA3377",
        "marker": "o",
        "linestyle": "--"
    },
    "WERM-unp-10fold": {
        "name": "WERM-unp-10fold",
        "label": "\\textbf{WERM}",
        "color": "black",
        "marker": "D",
        "linestyle": "-"
    }
}
methods_to_evaluate = [method_catalog[name].copy() for name in FINAL_METHOD_NAMES]

# Create a dictionary for easy lookup within run_single_trial
method_configs = {method['name']: method for method in methods_to_evaluate}
method_names_list = [m['name'] for m in methods_to_evaluate] # List of just names


# Results Storage
results = {
    'successful_trials': {method['name']: np.zeros(len(delta_list), dtype=int) for method in methods_to_evaluate},
    'true_chgpts': [],
    'n_values': [],
    'delta_values': [] # Renamed from noise_scale_values
}

for method in methods_to_evaluate:
    method_name = method['name']
    results[f'haus_{method_name}'] = np.full((len(delta_list), num_trials), np.nan)
    results[f'size_{method_name}'] = np.full((len(delta_list), num_trials), np.nan)
    results[f'runtime_{method_name}'] = np.full((len(delta_list), num_trials), np.nan)
    results[f'chgpts_{method_name}'] = []

# Least-squares fitting; retain absolute-error CV scoring.
M = lambda u, v: jnp.power(u - v, 2)

M_scorer = lambda u, v: jnp.abs(u - v) # Always use the absolute value loss for CV scoring!

# Gamma candidates for cross-validation of WERM
gamma_candidates = np.logspace(-1, 2, 5)

# --- Parallel Execution ---
tasks = [(i, delta, j) for i, delta in enumerate(delta_list) for j in range(num_trials)]

try:
    parallel_results = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(run_single_trial)(
            params=task,
            p=p,
            L=L,
            fixed_noise_scale=fixed_noise_scale, # Pass fixed noise_scale
            fractional_chgpt_locations=fractional_chgpt_locations,
            M=M,
            gamma_candidates=gamma_candidates,
            method_configs=method_configs
        ) for task in tqdm(tasks, desc="Running simulation trials", ncols=100)
    )
except Exception as e:
    print(f"Simulation failed with error: {str(e)}")
    raise

# --- Aggregate Results ---

for result_idx, (i, j, trial_result) in enumerate(parallel_results):
    for method_obj in methods_to_evaluate:
        method_name = method_obj['name']
        if trial_result['methods_status'][method_name]:
            results[f'haus_{method_name}'][i, j] = trial_result[f'haus_{method_name}']
            results[f'size_{method_name}'][i, j] = trial_result[f'size_{method_name}']
            results[f'runtime_{method_name}'][i, j] = trial_result[f'runtime_{method_name}']
            results['successful_trials'][method_name][i] += 1
        results[f'chgpts_{method_name}'].append((i, trial_result[f'chgpts_{method_name}']))

    results['true_chgpts'].append((i, trial_result['true_chgpts']))
    results['n_values'].append((i, trial_result['n']))
    results['delta_values'].append((i, delta_list[i])) # Store delta

results['trial_records'] = parallel_results
results['summaries'] = []
for setting_idx, setting in enumerate(delta_list):
    setting_records = [record for i, _, record in parallel_results if i == setting_idx]
    for method_obj in methods_to_evaluate:
        method_name = method_obj['name']
        summary = _summarize_setting(
            [record[f'haus_{method_name}'] for record in setting_records],
            [record[f'size_{method_name}'] for record in setting_records],
            [record[f'status_{method_name}'] for record in setting_records],
        )
        summary.update({'method': method_name, 'setting': float(setting)})
        results['summaries'].append(summary)
        print(method_name, f"delta={setting:.3g}", summary)

# Save this final experiment under an isolated, stable name.
FINAL_RESULT_FILE = "results/M1_upsample_final_alpha10_dense_gaussian_ls_seed46_100trials.pkl"
if SMOKE_MODE:
    FINAL_RESULT_FILE = FINAL_RESULT_FILE.replace(".pkl", "_smoke.pkl")
with open(FINAL_RESULT_FILE, "wb") as handle:
    pickle.dump({"results": results, "params": {
        "p": p, "L": L, "delta_list": delta_list,
        "fixed_noise_scale": fixed_noise_scale, "num_trials": num_trials,
        "fractional_chgpt_locations": fractional_chgpt_locations,
        "n_jobs": n_jobs, "successful_trials_counts": results["successful_trials"],
        "methods_evaluated": methods_to_evaluate, "random_seed": random_seed,
    }}, handle)
print(f"Saved final results to {FINAL_RESULT_FILE}")


Running simulation trials:   0%|                                              | 0/1 [00:00<?, ?it/s]

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Running McScan with automatic changepoint estimation.
Estimated changepoints from McScan:  [181. 213.]
\Delta: 2, n: 50, Lmin: 1, Lmax: 1
\Delta: 2, n: 50, Lmin: 1, Lmax: 1
\Delta: 2, n: 50, Lmin: 1, Lmax: 1


100%|██████████| 1/1 [00:00<00:00, 2207.53it/s]


\Delta: 2, n: 50, Lmin: 1, Lmax: 1
\Delta: 2, n: 50, Lmin: 1, Lmax: 1
\Delta: 2, n: 50, Lmin: 1, Lmax: 1


100%|██████████| 1/1 [00:00<00:00, 2231.01it/s]


... (60 outputs left out)

\Delta: 2, n: 50, Lmin: 4, Lmax: 4


100%|██████████| 1/1 [00:00<00:00, 13.87it/s]


\Delta: 2, n: 50, Lmin: 1, Lmax: 1
McScan delta=3 {'mean_haus': 0.295, 'median_haus': 0.295, 'empty_estimate_rate': 0.0, 'not_applicable_rate': 0.0, 'mean_size': 2.0, 'method': 'McScan', 'setting': 3.0}
WERM-unp-10fold delta=3 {'mean_haus': nan, 'median_haus': nan, 'empty_estimate_rate': 1.0, 'not_applicable_rate': 0.0, 'mean_size': 0.0, 'method': 'WERM-unp-10fold', 'setting': 3.0}
Saved final results to results/M1_upsample_final_alpha10_dense_gaussian_ls_seed46_100trials_smoke.pkl


100%|██████████| 1/1 [00:00<00:00, 2548.18it/s]


In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

plt.switch_backend("module://matplotlib_inline.backend_inline")
plt.rcParams.update({
    'pgf.texsystem': 'pdflatex',
    'pgf.preamble': r'\usepackage{amsmath}\usepackage{amsfonts}\usepackage{amssymb}\usepackage{siunitx}',
    'font.size': 14,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman', 'Times New Roman', 'DejaVu Serif'],
    'text.usetex': True,
    'axes.linewidth': 1.5,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.fontsize': 14,
    'legend.frameon': True,
    'legend.fancybox': True,
    'legend.edgecolor': 'black',
    'legend.framealpha': 0.95,
    'lines.linewidth': 4.0,
    'lines.markersize': 8,
    'figure.dpi': 72,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

FINAL_RESULT_FILE = "results/M1_upsample_final_alpha10_dense_gaussian_ls_seed46_100trials_smoke.pkl"
if os.environ.get("WERM_SMOKE") == "1":
    FINAL_RESULT_FILE = FINAL_RESULT_FILE.replace(".pkl", "_smoke.pkl")
with open(FINAL_RESULT_FILE, "rb") as handle:
    _payload = pickle.load(handle)
results = _payload["results"]
_params = _payload["params"]
p, L = _params["p"], _params["L"]
delta_list = np.asarray(_params["delta_list"], dtype=float)
fractional_chgpt_locations = np.asarray(_params["fractional_chgpt_locations"])

methods_to_evaluate = [
    dict(name='McScan', label=r'McScan', color='#AA3377', marker='o', linestyle='--'),
    dict(name='WERM-unp-10fold', label=r'\textbf{WERM}', color='black', marker='D', linestyle='-'),
]
method_names_list = [method['name'] for method in methods_to_evaluate]
for method_name in method_names_list:
    _size = np.asarray(results[f'size_{method_name}'], dtype=float)
    _haus = np.asarray(results[f'haus_{method_name}'], dtype=float)
    results[f'haus_{method_name}'] = np.where(_size > 0, _haus, np.nan)

def get_asymmetric_errors(data):
    data = np.asarray(data, dtype=float)
    means = np.nanmean(data, axis=1)
    p25 = np.nanpercentile(data, 25, axis=1)
    p75 = np.nanpercentile(data, 75, axis=1)
    return [np.abs(means - p25), np.abs(p75 - means)]

def _plot_asymmetric_metric(metric, ylabel, reference_y=None, y_limits=None):
    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    for method in methods_to_evaluate:
        values = np.asarray(results[f'{metric}_{method["name"]}'], dtype=float)
        means = np.nanmean(values, axis=1)
        ax.errorbar(
            delta_list, means, yerr=get_asymmetric_errors(values),
            label=method['label'], marker=method['marker'], ls=method['linestyle'],
            color=method['color'], capsize=4, capthick=2, markersize=8,
            linewidth=3.5, markeredgewidth=2, markeredgecolor='white',
        )
    if reference_y is not None:
        ax.axhline(reference_y, color='gray', linestyle='-', linewidth=2.5,
                   alpha=0.8, label=r'True no. change points')
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    ax.set_xlabel(r'$\delta = n/p$', fontsize=22)
    ax.set_ylabel(ylabel, fontsize=22)
    ax.set_xticks(delta_list[::2])
    ax.set_xticklabels([f'{value:.0f}' for value in delta_list[::2]], fontsize=22)
    ax.tick_params(axis='both', which='major', width=1.5, length=6, labelsize=22)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.legend(loc='best', prop={'size': 17}, fancybox=True, framealpha=0.4)
    fig.tight_layout()
    plt.show()
    return fig, ax

def plot_size_asymmetric_errors():
    return _plot_asymmetric_metric(
        'size', r'No. change points',
        reference_y=len(fractional_chgpt_locations), y_limits=(0, L),
    )

def plot_location_asymmetric_errors():
    return _plot_asymmetric_metric('haus', r'Hausdorff dist./$n$')

def plot_runtime_asymmetric_errors():
    return _plot_asymmetric_metric('runtime', r'Runtime (seconds)', y_limits=(0, None))

x_y_label_fontsize = 22

# --- Distribution Plotting Functions -----------------------------------------
#
# Three display modes are supported:
#   1. "box_jitter"      : standard boxplot + all observations (recommended)
#   2. "interval_jitter" : 5--95%, IQR, median + all observations
#   3. "violin_jitter"   : violin + IQR/median + all observations
#
# A box with Q1 == Q3 is statistically supposed to collapse.  We therefore do
# not artificially change its quartiles.  Instead, collapsed IQRs are reinforced
# with a thick horizontal line and, optionally, a hollow-square glyph.

from matplotlib.lines import Line2D


_DISTRIBUTION_STYLES = {"box_jitter", "interval_jitter", "violin_jitter"}
_COLLAPSED_STYLES = {"line", "line_and_marker", "none"}


def _finite_1d(values):
    """Return a flattened float array containing only finite observations."""
    values = np.asarray(values, dtype=float).ravel()
    return values[np.isfinite(values)]


def _distribution_summary(values):
    """Compute the summaries used by all distribution-plotting modes."""
    values = _finite_1d(values)
    if values.size == 0:
        return None

    q05, q25, median, q75, q95 = np.percentile(
        values, [5, 25, 50, 75, 95]
    )
    data_scale = max(
        1.0,
        float(np.max(np.abs(values))),
        float(np.ptp(values)),
    )
    collapsed = np.isclose(
        q25,
        q75,
        rtol=1e-10,
        atol=1e-12 * data_scale,
    )

    return {
        "n": int(values.size),
        "n_unique": int(np.unique(values).size),
        "min": float(np.min(values)),
        "q05": float(q05),
        "q25": float(q25),
        "median": float(median),
        "q75": float(q75),
        "q95": float(q95),
        "max": float(np.max(values)),
        "iqr": float(q75 - q25),
        "collapsed": bool(collapsed),
    }


def _warn(message):
    """Use the project logger when it supports warnings; otherwise print."""
    logger = globals().get("main_logger")
    warning_method = getattr(logger, "warning", None)
    if callable(warning_method):
        warning_method(message)
    else:
        print(f"WARNING: {message}")


def _jittered_x(position, n, rng, jitter_width):
    """Generate reproducible horizontal jitter without changing y values."""
    if n <= 1 or jitter_width <= 0:
        return np.full(n, position, dtype=float)
    return position + rng.uniform(-jitter_width, jitter_width, size=n)


def _draw_raw_points(
    ax,
    values,
    position,
    color,
    rng,
    jitter_width=0.18,
    point_size=22,
    point_alpha=0.16,
):
    """Overlay all observations using x-only jitter."""
    x = _jittered_x(position, values.size, rng, jitter_width)
    ax.scatter(
        x,
        values,
        s=point_size,
        facecolor=color,
        edgecolor="none",
        alpha=point_alpha,
        zorder=4,
        clip_on=True,
        label="_nolegend_",
    )


def _draw_collapsed_iqr(
    ax,
    position,
    median,
    box_width,
    collapsed_style="line_and_marker",
):
    """
    Make a zero-IQR group visible without inventing a nonzero statistical box.

    The optional square is a display glyph only; it does not encode an interval.
    """
    if collapsed_style not in _COLLAPSED_STYLES:
        raise ValueError(
            f"collapsed_style must be one of {sorted(_COLLAPSED_STYLES)}, "
            f"got {collapsed_style!r}."
        )
    if collapsed_style == "none":
        return

    half_width = 0.43 * box_width
    ax.plot(
        [position - half_width, position + half_width],
        [median, median],
        color="black",
        linewidth=3.4,
        solid_capstyle="butt",
        zorder=7,
        label="_nolegend_",
    )

    if collapsed_style == "line_and_marker":
        ax.scatter(
            [position],
            [median],
            marker="s",
            s=72,
            facecolor="none",
            edgecolor="black",
            linewidth=1.25,
            zorder=8,
            label="_nolegend_",
        )


def _draw_box_jitter_group(
    ax,
    values,
    position,
    color,
    rng,
    summary,
    box_width,
    jitter_width,
    point_size,
    point_alpha,
    collapsed_style,
):
    """Draw one honest boxplot and overlay all observations."""
    bp = ax.boxplot(
        [values],
        positions=[position],
        widths=box_width,
        patch_artist=True,
        showfliers=False,       # every observation is shown by the scatter layer
        whis=1.5,
        autorange=True,         # if IQR=0, extend whiskers to min/max
        manage_ticks=False,
        medianprops={
            "color": "black",
            "linewidth": 2.3,
            "zorder": 6,
        },
        boxprops={
            "color": "black",
            "linewidth": 1.35,
            "zorder": 2,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.25,
            "zorder": 2,
        },
        capprops={
            "color": "black",
            "linewidth": 1.25,
            "zorder": 2,
        },
    )

    patch = bp["boxes"][0]
    patch.set_facecolor(color)
    patch.set_alpha(0.28)
    patch.set_edgecolor("black")

    _draw_raw_points(
        ax=ax,
        values=values,
        position=position,
        color=color,
        rng=rng,
        jitter_width=jitter_width,
        point_size=point_size,
        point_alpha=point_alpha,
    )

    # Reinforce the statistical line when the box height is genuinely zero.
    if summary["collapsed"]:
        _draw_collapsed_iqr(
            ax=ax,
            position=position,
            median=summary["median"],
            box_width=box_width,
            collapsed_style=collapsed_style,
        )


def _draw_interval_summary(
    ax,
    position,
    color,
    summary,
    box_width,
    collapsed_style,
):
    """Draw 5--95%, IQR, and median summaries."""
    cap_half_width = 0.14 * box_width

    # Thin 5--95% interval and end caps.
    ax.vlines(
        position,
        summary["q05"],
        summary["q95"],
        color="black",
        linewidth=1.35,
        zorder=5,
    )
    ax.hlines(
        [summary["q05"], summary["q95"]],
        position - cap_half_width,
        position + cap_half_width,
        color="black",
        linewidth=1.35,
        zorder=5,
    )

    # Thick IQR interval. It correctly becomes a point/line when IQR=0.
    ax.vlines(
        position,
        summary["q25"],
        summary["q75"],
        color=color,
        linewidth=8.0,
        alpha=0.58,
        zorder=5,
    )

    # Median remains visible even when every interval collapses.
    ax.scatter(
        [position],
        [summary["median"]],
        marker="D",
        s=46,
        facecolor=color,
        edgecolor="black",
        linewidth=0.9,
        zorder=7,
        label="_nolegend_",
    )

    if summary["collapsed"]:
        _draw_collapsed_iqr(
            ax=ax,
            position=position,
            median=summary["median"],
            box_width=box_width,
            collapsed_style=collapsed_style,
        )


def _draw_interval_jitter_group(
    ax,
    values,
    position,
    color,
    rng,
    summary,
    box_width,
    jitter_width,
    point_size,
    point_alpha,
    collapsed_style,
):
    """Draw raw observations together with quantile intervals and the median."""
    _draw_raw_points(
        ax=ax,
        values=values,
        position=position,
        color=color,
        rng=rng,
        jitter_width=jitter_width,
        point_size=point_size,
        point_alpha=point_alpha,
    )
    _draw_interval_summary(
        ax=ax,
        position=position,
        color=color,
        summary=summary,
        box_width=box_width,
        collapsed_style=collapsed_style,
    )


def _draw_violin_jitter_group(
    ax,
    values,
    position,
    color,
    rng,
    summary,
    box_width,
    jitter_width,
    point_size,
    point_alpha,
    collapsed_style,
):
    """
    Draw a violin/raincloud-style display plus points and quantile summaries.

    The violin is omitted when there are fewer than two distinct observations,
    because a kernel density is then undefined/uninformative.
    """
    if values.size >= 2 and summary["n_unique"] >= 2:
        parts = ax.violinplot(
            [values],
            positions=[position],
            widths=1.15 * box_width,
            showmeans=False,
            showmedians=False,
            showextrema=False,
            points=100,
            bw_method="scott",
        )
        body = parts["bodies"][0]
        body.set_facecolor(color)
        body.set_edgecolor("black")
        body.set_linewidth(1.0)
        body.set_alpha(0.20)
        body.set_zorder(1)

    _draw_raw_points(
        ax=ax,
        values=values,
        position=position,
        color=color,
        rng=rng,
        jitter_width=jitter_width,
        point_size=point_size,
        point_alpha=point_alpha,
    )
    _draw_interval_summary(
        ax=ax,
        position=position,
        color=color,
        summary=summary,
        box_width=box_width,
        collapsed_style=collapsed_style,
    )


def _format_distribution_diagnostic(metric_label, method_name, delta, summary):
    return (
        f"{metric_label}; method={method_name}; delta^2={delta:g}: "
        f"n={summary['n']}, unique={summary['n_unique']}, "
        f"min={summary['min']:.6g}, Q1={summary['q25']:.6g}, "
        f"median={summary['median']:.6g}, Q3={summary['q75']:.6g}, "
        f"max={summary['max']:.6g}, IQR={summary['iqr']:.6g}, "
        f"collapsed={summary['collapsed']}"
    )


def _plot_grouped_distributions(
    current_delta_list,
    current_results,
    methods_to_evaluate,
    metric_prefix,
    metric_label,
    y_label,
    filename_base,
    filename_metric,
    plot_style="box_jitter",
    collapsed_style="line_and_marker",
    random_seed=2026,
    box_width=0.72,
    jitter_width=0.20,
    point_size=22,
    point_alpha=0.16,
    diagnostics=False,
    show_legend=True,
    legend_loc="upper left",
    y_limits=None,
    y_ticks=None,
    reference_y=None,
    reference_label=None,
    x_label=r'$\delta = n/p$',
    tick_label_format=".0f",
    tick_stride=1,
):
    """Shared implementation for location, size, and runtime distributions."""
    if plot_style not in _DISTRIBUTION_STYLES:
        raise ValueError(
            f"plot_style must be one of {sorted(_DISTRIBUTION_STYLES)}, "
            f"got {plot_style!r}."
        )
    if collapsed_style not in _COLLAPSED_STYLES:
        raise ValueError(
            f"collapsed_style must be one of {sorted(_COLLAPSED_STYLES)}, "
            f"got {collapsed_style!r}."
        )

    fig, ax = plt.subplots(1, 1, figsize=(5.0, 4.0))
    rng = np.random.default_rng(random_seed)

    plot_labels = [method["label"] for method in methods_to_evaluate]
    plot_colors = [method["color"] for method in methods_to_evaluate]
    methods = [method["name"] for method in methods_to_evaluate]

    spacing = len(methods) + 1
    diagnostic_records = []

    for delta_idx, delta in enumerate(current_delta_list):
        for method_idx, method_name in enumerate(methods):
            data = current_results.get(f"{metric_prefix}_{method_name}")
            if data is None or data.ndim < 2 or data.shape[0] <= delta_idx:
                _warn(
                    f"No {metric_label} data for {method_name} at "
                    f"delta^2={delta:g}; skipping this group."
                )
                continue

            values = _finite_1d(data[delta_idx, :])
            if values.size == 0:
                _warn(
                    f"No finite {metric_label} observations for {method_name} "
                    f"at delta^2={delta:g}; skipping this group."
                )
                continue

            summary = _distribution_summary(values)
            position = delta_idx * spacing + method_idx
            color = plot_colors[method_idx]

            record = {
                "metric": metric_label,
                "method": method_name,
                "delta_value": float(delta),
                **summary,
            }
            diagnostic_records.append(record)
            if diagnostics:
                print(
                    _format_distribution_diagnostic(
                        metric_label, method_name, delta, summary
                    )
                )

            draw_kwargs = dict(
                ax=ax,
                values=values,
                position=position,
                color=color,
                rng=rng,
                summary=summary,
                box_width=box_width,
                jitter_width=jitter_width,
                point_size=point_size,
                point_alpha=point_alpha,
                collapsed_style=collapsed_style,
            )

            if plot_style == "box_jitter":
                _draw_box_jitter_group(**draw_kwargs)
            elif plot_style == "interval_jitter":
                _draw_interval_jitter_group(**draw_kwargs)
            else:
                _draw_violin_jitter_group(**draw_kwargs)

    # Optional horizontal truth/reference line.
    if reference_y is not None:
        ax.axhline(
            y=reference_y,
            color="black",
            linestyle="-",
            linewidth=2.4,
            alpha=0.75,
            zorder=0,
            label="_nolegend_",
        )

    delta_positions = [
        i * spacing + (len(methods) - 1) / 2
        for i in range(len(current_delta_list))
    ]
    tick_indices = np.arange(0, len(current_delta_list), tick_stride, dtype=int)
    ax.set_xticks([delta_positions[i] for i in tick_indices])
    ax.set_xticklabels(
        [format(float(current_delta_list[i]), tick_label_format) for i in tick_indices]
    )
    ax.set_xlabel(x_label, fontsize=x_y_label_fontsize)
    ax.set_ylabel(y_label, fontsize=x_y_label_fontsize)

    if y_limits is not None:
        ax.set_ylim(*y_limits)
    if y_ticks is not None:
        ax.set_yticks(y_ticks)

    if show_legend:
        legend_elements = [
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="none",
                markersize=8,
                markerfacecolor=plot_colors[i],
                markeredgecolor="black",
                markeredgewidth=0.8,
                alpha=0.75,
                label=plot_labels[i],
            )
            for i in range(len(methods))
        ]
        if reference_y is not None and reference_label is not None:
            legend_elements.append(
                Line2D(
                    [0],
                    [0],
                    color="black",
                    linewidth=2.4,
                    label=reference_label,
                )
            )
        ax.legend(
            handles=legend_elements,
            loc=legend_loc,
            fontsize=16,
            frameon=True,
            edgecolor="black",
            framealpha=0.95,
        )

    ax.tick_params(
        axis="both",
        which="major",
        width=1.5,
        length=6,
        labelsize=x_y_label_fontsize,
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)

    # Keep the historical filenames for the recommended box+jitter mode.
    style_suffix = {
        "box_jitter": "boxplots",
        "interval_jitter": "interval_jitter",
        "violin_jitter": "violin_jitter",
    }[plot_style]
    output_stem = f"{filename_base}_{filename_metric}_{style_suffix}"

    fig.tight_layout()



    plt.show()

    return fig, ax, diagnostic_records


def plot_location_boxplots(
    current_delta_list,
    current_results,
    methods_to_evaluate,
    filename_base,
    plot_style="box_jitter",
    collapsed_style="line_and_marker",
    diagnostics=False,
):
    """
    Plot the Hausdorff-error distributions.

    Recommended publication mode: plot_style="box_jitter".
    Clean alternative:             plot_style="interval_jitter".
    """
    return _plot_grouped_distributions(
        current_delta_list=current_delta_list,
        current_results=current_results,
        methods_to_evaluate=methods_to_evaluate,
        metric_prefix="haus",
        metric_label="Hausdorff distance / n",
        y_label=r'Hausdorff dist./$n$',
        filename_base=filename_base,
        filename_metric="location",
        plot_style=plot_style,
        collapsed_style=collapsed_style,
        diagnostics=diagnostics,
        show_legend=False,
        tick_label_format=".0f",
        tick_stride=2,
        x_label=r'$\delta = n/p$',
    )


def plot_size_boxplots(
    current_delta_list,
    current_results,
    methods_to_evaluate,
    L,
    fractional_chgpt_locations,
    filename_base,
    plot_style="box_jitter",
    collapsed_style="line_and_marker",
    diagnostics=False,
):
    """
    Plot estimated changepoint counts.

    Because this response is discrete, box+jitter or interval+jitter is preferable
    to a violin plot, whose kernel smoothing can imply nonexistent values.
    """
    return _plot_grouped_distributions(
        current_delta_list=current_delta_list,
        current_results=current_results,
        methods_to_evaluate=methods_to_evaluate,
        metric_prefix="size",
        metric_label="Number of change points",
        y_label=r'No. change points',
        filename_base=filename_base,
        filename_metric="size",
        plot_style=plot_style,
        collapsed_style=collapsed_style,
        diagnostics=diagnostics,
        show_legend=True,
        legend_loc="upper left",
        y_limits=(-0.25, L + 0.25),
        y_ticks=np.arange(0, L + 1, dtype=int),
        reference_y=len(fractional_chgpt_locations),
        reference_label=r'\textbf{True no. change points}',
        tick_label_format=".0f",
        x_label=r'$\delta = n/p$',
    )


def plot_runtime_boxplots(
    current_delta_list,
    current_results,
    methods_to_evaluate,
    filename_base,
    plot_style="box_jitter",
    collapsed_style="line_and_marker",
    diagnostics=False,
):
    """Plot runtime distributions using any supported display mode."""
    return _plot_grouped_distributions(
        current_delta_list=current_delta_list,
        current_results=current_results,
        methods_to_evaluate=methods_to_evaluate,
        metric_prefix="runtime",
        metric_label="Runtime (seconds)",
        y_label=r'Runtime (seconds)',
        filename_base=filename_base,
        filename_metric="runtime",
        plot_style=plot_style,
        collapsed_style=collapsed_style,
        diagnostics=diagnostics,
        show_legend=True,
        legend_loc="upper left",
        y_limits=(0, None),
        tick_label_format=".0f",
        x_label=r'$\delta = n/p$',
    )



print('Dense M1b (alpha=1.0, Gaussian noise): M3-final-style panels')
print("1. Location error: asymmetric-error line plot")
plot_location_asymmetric_errors()
print("2. Location error: boxplot + deterministic raw-point jitter")
plot_location_boxplots(
    delta_list, results, methods_to_evaluate, "M1_dense_final",
    plot_style="box_jitter", collapsed_style="line_and_marker",
)
print("3. Number of change points: asymmetric-error line plot")
plot_size_asymmetric_errors()
print("4. Number of change points: boxplot + deterministic raw-point jitter")
plot_size_boxplots(
    delta_list, results, methods_to_evaluate, L, fractional_chgpt_locations,
    "M1_dense_final", plot_style="box_jitter", collapsed_style="line_and_marker",
)
print("5. Runtime: asymmetric-error line plot")
plot_runtime_asymmetric_errors()
print("6. Runtime: boxplot + deterministic raw-point jitter")
plot_runtime_boxplots(
    delta_list, results, methods_to_evaluate, "M1_dense_final",
    plot_style="box_jitter", collapsed_style="line_and_marker",
)


FileNotFoundError: [Errno 2] No such file or directory: 'results/M1_upsample_final_alpha10_dense_gaussian_ls_seed46_100trials_smoke_smoke.pkl'